# Augment: Intercity Passenger Rail Service Station Performance Metrics

This notebook augments the quarterly [Amtrak](https://www.amtrak.com/home.html) station performance
metrics with additional information about each station. The dataset is sourced from the US
Department of Transportation (DOT), Bureau of Transportation Statistics (BTS), ArcGIS online
[Amtrak Stations](https://geodata.bts.gov/datasets/1ed62a9f46304679aaa396bed4c8565a_0/about) layer.
The dataset contains information about the location of each station, including the station name,
city, state, and geo coordinates.

### Variable names

A number of variable names in this project leverage the following abbreviations. The naming
strategy is to strike a balance between brevity and readability:

* `amtk`: Amtrak (reporting mark)
* `chrt`: chart
* `cols`: columns
* `const`: constant
* `cwd`: current working directory
* `eb`: eastbound direction of travel
* `lm`: linear model
* `mi`: miles
* `mm`: minutes (ISO 8601)
* `nb`: northbound direction of travel
* `psgr`: passenger
* `qtr`: quarter
* `rte`: route
* `sb`: southbound direction of travel
* `stats`: summary statistics
* `stn`: station
* `stns`: stations
* `svc`: service
* `trn`: train
* `wb`: westbound direction of travel

In [1]:
import json
import numpy as np
import pandas as pd
import pathlib as pl
import re
import tomllib as tl

import fra_amtrak.amtk_frame as frm

# Set random seed
rdg = np.random.default_rng(24)

## 1.0 Read files

### 1.1 Resolve paths

Instantiate instances of `pathlib.Path` to represent absolute paths to the `data/interim` and `data/processed` directories.

In [2]:
parent_path = pl.Path.cwd()  # current working directory
parent_path

data_raw_path = parent_path.joinpath("data", "raw")
data_interim_path = parent_path.joinpath("data", "interim")
data_processed_path = parent_path.joinpath("data", "processed")

### 1.2 Load constants

Load a companion [TOML](https://toml.io/en/) file containing constants.

In [3]:
filepath = parent_path.joinpath("notebook.toml")
with open(filepath, "rb") as file_obj:
    const = tl.load(file_obj)

# Access constants
COLS = const["columns"]

filepath = data_interim_path.joinpath("station_performance_metrics-v1p1.csv")
stations = pd.read_csv(filepath)

### 1.3 Retrieve performance data

In [4]:
filepath = data_interim_path.joinpath("station_performance_metrics-v1p1.csv")
stations = pd.read_csv(filepath)

### 1.4 Review the `DataFrame`

In [5]:
stations.shape

(68412, 15)

In [6]:
stations.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 68412 entries, 0 to 68411
Data columns (total 15 columns):
 #   Column                                  Non-Null Count  Dtype  
---  ------                                  --------------  -----  
 0   Fiscal Year                             68412 non-null  int64  
 1   Fiscal Quarter                          68412 non-null  int64  
 2   Service Line                            68412 non-null  object 
 3   Service                                 68412 non-null  object 
 4   Sub Service                             68412 non-null  object 
 5   Train Number                            68412 non-null  int64  
 6   Arrival Station Code                    68412 non-null  object 
 7   Arrival Station                         68412 non-null  object 
 8   State                                   68412 non-null  object 
 9   Division                                68354 non-null  object 
 10  Region                                  68354 non-null  ob

In [7]:
stations.head(3)

,Fiscal Year,Fiscal Quarter,Service Line,Service,Sub Service,Train Number,Arrival Station Code,Arrival Station,State,Division,Region,Country,Total Detraining Customers,Late Detraining Customers,Late Detraining Customers Avg Min Late
0,2024,3,Long Distance,Auto Train,Auto Train,52,LOR,Lorton (Auto Train),Virginia,South Atlantic,South,United States,42445,23316,95.0
1,2024,3,Long Distance,Auto Train,Auto Train,53,SFA,Sanford (Auto Train),Florida,South Atlantic,South,United States,28034,18439,91.0
2,2024,3,Long Distance,California Zephyr,California Zephyr,5,BRL,Burlington,Iowa,West North Central,Midwest,United States,557,223,54.0


## 2.0 Add route miles

Every named train is associated with a route that Amtrak measures in miles. The route miles data was sourced
from the FRA's
[_Methodology Report for the Performance and Service Quality of Intercity Passenger Train Operations_](https://railroads.dot.gov/sites/fra.dot.gov/files/2024-08/Methodology%20Report_FY24Q3_web.pdf) (FY 2024 v.2), pp. 12-15.

### 2.1 Retrieve data

In [8]:
with open(data_processed_path.joinpath("amtk_sub_services.json"), "r") as file:
    amtk_sub_svcs = json.load(file)

route_miles = [
    {"Route": route["sub service"], "Route Miles": sum([host["miles"] for host in route["hosts"]])}
    for route in amtk_sub_svcs
]

# Create DataFrame
route_miles = pd.DataFrame.from_dict(route_miles, orient="columns")
route_miles

,Route,Route Miles
0,Auto Train,914
1,California Zephyr,2408
2,Capitol Ltd,788
3,Cardinal,1140
4,City Of New Orleans,930
5,Coast Starlight,1388
6,Crescent,1367
7,Empire Builder,2560
8,Lake Shore Ltd,1255
9,Palmetto,885


### 2.2 Combine data [1 pt]

Add `route_miles` to the `stations` `DataFrame`. Once the data is combined, move the `route_miles`
column from the last position to the fifth (`5th`) position in `stations`. Drop any redundant
columns after reordering the columns.

In [9]:
# YOUR CODE HERE
# 1. Merge data
stations = stations.merge(route_miles, left_on="Sub Service", right_on="Route", how="left")

# 2. Ambil list urutan kolom
cols = stations.columns.tolist()

# 3. Pindahkan "Route Miles" ke posisi indeks 5 (posisi ke-6)
cols.remove("Route Miles")
cols.insert(5, "Route Miles")

# 4. Terapkan urutan dan buang kolom redundan
stations = stations[cols].drop(columns=["Route"])

In [10]:
#hidden tests are within this cell

## 3.0 Add location data

The Bureau of Transportation Statistics (BTS) maintains an [Amtrak stations](https://data-usdot.opendata.arcgis.com/datasets/amtrak-stations/about) dataset that provides mapping (i.e., location) information.

### 3.1 Retrieve data

In [11]:
filepath = data_raw_path.joinpath("NTAD_Amtrak_Stations_-3056704789218436106.csv")
ntad_stations = pd.read_csv(filepath)

### 3.2 Filter data [1 pt]

Filter out all bus stations and reset the index.

In [12]:
# YOUR CODE HERE
# Membuang semua baris yang kolom StaType-nya mengandung kata "Bus"
ntad_stations = ntad_stations[~ntad_stations["StaType"].str.contains("Bus", case=False, na=False)].reset_index(drop=True)

In [13]:
#hidden tests are within this cell

### 3.3 Drop columns [1 pt]

Drop the following columns. They are not required for the analysis.

* OBJECTID
* StnType
* State
* Name
* StationName
* StationFacilityName
* StationAliases
* DateModif
* x
* y

 Retain only the "StaType", "ZipCode", "Address2", "Address1", "Code", "lon", and "lat" columns.

In [15]:
# YOUR CODE HERE
# List kolom yang mau dihapus sesuai instruksi
cols_to_drop = [
    "OBJECTID", "StnType", "State", "Name", "StationName", 
    "StationFacilityName", "StationAliases", "DateModif", "x", "y"
]
ntad_stations = ntad_stations.drop(columns=cols_to_drop)

In [16]:
#hidden tests are within this cell

## 4.0 Clean data

### 4.1 Blank and missing values

No empty or missing values it appears.

In [17]:
# Combined condition to check for empty strings or NaN
mask = (ntad_stations == "") | pd.isna(ntad_stations)
empty_nan_values = ntad_stations.columns[mask.any()]
empty_nan_values

# Count empty or NaN values
# empty_nan_counts = ntad_stations[empty_nan_values].apply(lambda x: x.isin(["", np.nan]).sum())
# empty_nan_counts

Index([], dtype='object')

### 4.2 Normalize strings

Trim each string value of leading/trailing spaces. Also search and remove unnecessary spaces in each string value based on the regular expression `re.Pattern` object. Call the function `frm.normalize_dataframe_strings()` to perform this operation.

#### 4.2.1 Locate suspect strings

As is illustrated below, the regex pattern to employ is `"\s{2,}"`.

In [18]:
# Locate extra spaces in all string columns
extra_spaces = ntad_stations.select_dtypes(include="object").apply(
    lambda x: x.str.contains(r"\s{2,}").sum()
)
extra_spaces

StaType     0
ZipCode     0
City        0
Address2    0
Address1    4
Code        0
dtype: int64

#### 4.2.2 Clean strings [1 pt]

In [19]:
# YOUR CODE HERE
ntad_stations = frm.normalize_dataframe_strings(ntad_stations, r"\s{2,}")

In [20]:
#hidden tests are within this cell

## 5.0 Manipulate data

### 5.1 Rename the columns

Note use of constants.

In [21]:
mapper = {
    "StaType": COLS["station_type"],
    "ZipCode": COLS["zip_code"],
    "City": COLS["city"],
    "Address2": COLS["address_02"],
    "Address1": COLS["address_01"],
    "Code": "Code",
    "lon": COLS["lon"],
    "lat": COLS["lat"],
}
ntad_stations.rename(columns=mapper, inplace=True)
ntad_stations.head(3)

,Arrival Station Type,ZIP Code,City,Address 02,Address 01,Code,Longitude,Latitude
0,Station Building (with waiting room),48801,Alma,,1105 Willow Run Drive,AAM,-84.644832,43.391720
1,Curbside Bus Stop only (no shelter),12211,Albany,,737 Albany Shaker Road,ABA,-73.809184,42.744491
2,Curbside Bus Stop only (no shelter),54421,Colby,,1210 North Division St.,ABB,-90.314667,44.928553


### 5.2 Reorder columns

:bulb: By convention, latitude is always listed before longitude.

In [22]:
columns = [
    "Code",
    COLS["station_type"],
    COLS["city"],
    COLS["address_01"],
    COLS["address_02"],
    COLS["zip_code"],
    COLS["lat"],
    COLS["lon"],
]
ntad_stations = ntad_stations.loc[:, columns]
ntad_stations.sample(n=7, random_state=rdg)

,Code,Arrival Station Type,City,Address 01,Address 02,ZIP Code,Latitude,Longitude
401,IRV,Station Building (with waiting room),Irvine,15215 Barranca Parkway,,92618,33.656771,-117.733695
134,CHS,Station Building (with waiting room),North Charleston,4565 Gaynor Avenue,North Charleston Transit Center,29405,32.875473,-79.998048
347,GVB,Platform with Shelter,Grover Beach,180 West Grand Avenue,,93433,35.121260,-120.629266
827,SHW,Curbside Bus Stop only (no shelter),Shawano,N4543 State Highway 22,,54166,44.753147,-88.615339
606,NCW,Station Building (with waiting room),North Conway,2760 White Mountain Highway,,03860,44.054485,-71.129461
790,RVR,Station Building (with waiting room),Richmond,7519 Staples Mill Road,,23228,37.617693,-77.496943
426,KIS,Station Building (with waiting room),Kissimmee,111 East Dakin Avenue,,34741,28.293835,-81.404234


## 6.0 Merge data [1 pt]

Merge `stations` and `ntad_stations`. Perform a __left join__ to retain all rows in the `stations` `DataFrame`, joining on the "Arrival Station Code" column in `stations` and the "Code" column in `ntad_stations`.

In [23]:
# YOUR CODE HERE
stations = stations.merge(
    ntad_stations, 
    left_on="Arrival Station Code", 
    right_on="Code", 
    how="left"
)

In [24]:
#hidden tests are within this cell

## 7.0 Check geo coordinates [1 pt]

Check for missing geo coordinates in the latitude and longitude columns in the merged DataFrame
named `stations`. Create a new `DataFrame` named `missing_coords` containing the filtered rows.
Limit the new `DataFrame` to the following columns:

* "Arrival Station Code"
* "Arrival Station"
* "State"
* "Latitude"
* "Longitude"

In [25]:
# YOUR CODE HERE
# 1. Tentukan list kolom yang diminta sesuai instruksi
cols_to_keep = [
    "Arrival Station Code", 
    "Arrival Station", 
    "State", 
    "Latitude", 
    "Longitude"
]

# 2. Filter baris yang Latitude atau Longitude-nya kosong (NaN), 
# lalu ambil hanya kolom-kolom yang ada di dalam list cols_to_keep
missing_coords = stations[stations["Latitude"].isna() | stations["Longitude"].isna()][cols_to_keep]

In [26]:
#hidden tests are within this cell

### 7.1 Missing geo coordinates

The BTS Amtrak stations dataset does not contain geo coordinates for the following stations:

* CBN: Canadian Border, NY
* FAL: Falmouth, ME
* MCI: Michigan City, IN

#### 7.1.1 CBN

This is not a physical station but an international border crossing in the vicinity of
Niagra Falls that features an exchange of US and Canadian train crews. The MCI
[Michigan City Station](https://en.wikipedia.org/wiki/Michigan_City_station) is a former Amtrak
station that was closed on 4 April 2022. The geo coordinates for the station can be obtained from
[Google Maps](https://www.google.com/maps/place/41%C2%B043'16.0%22N+86%C2%B054'20.0%22W/@41.721111,-86.905556,15z/data=!4m4!3m3!8m2!3d41.721111!4d-86.905556?hl=en&entry=ttu&g_ep=EgoyMDI0MTAyOS4wIKXMDSoASAFQAw%3D%3D).

#### 7.1.2 FAL

A special event stop for the Amtrak [Downeaster](https://www.amtrak.com/downeaster-train)
in support of the _The Live + Work in Maine Open Golf Tournament_ held at the
[Falmouth Country Club](https://www.falmouthcc.org/) during June 24-27, 2021 and June 23-26, 2022
(source: http://www.trainweb.org/usarail/falmouth.htm).

FAL row values can be updated with the following information:

Muirfield Road at Railroad Crossing \
Falmouth, ME 04105 \
Latitude: `43.769600`, Longitude: `-70.259500`

In [27]:
values = ("Falmouth", "Muirfield Road at Railroad Crossing", "04105", 43.769600, -70.259500)
mask = stations[COLS["station_code"]] == "FAL"
stations.loc[
    mask,
    [COLS["city"], COLS["address_01"], COLS["zip_code"], COLS["lat"], COLS["lon"]],
] = values
stations[mask]

,Fiscal Year,Fiscal Quarter,Service Line,Service,Route Miles,Sub Service,Train Number,Arrival Station Code,Arrival Station,State,...,Late Detraining Customers,Late Detraining Customers Avg Min Late,Code,Arrival Station Type,City,Address 01,Address 02,ZIP Code,Latitude,Longitude
51414,2022,3,State Supported,Downeaster,145,Downeaster,681,FAL,Falmouth,Maine,...,0,NaN,NaN,NaN,Falmouth,Muirfield Road at Railroad Crossing,NaN,04105,43.7696,-70.2595
51426,2022,3,State Supported,Downeaster,145,Downeaster,682,FAL,Falmouth,Maine,...,0,NaN,NaN,NaN,Falmouth,Muirfield Road at Railroad Crossing,NaN,04105,43.7696,-70.2595
51448,2022,3,State Supported,Downeaster,145,Downeaster,684,FAL,Falmouth,Maine,...,0,NaN,NaN,NaN,Falmouth,Muirfield Road at Railroad Crossing,NaN,04105,43.7696,-70.2595
51471,2022,3,State Supported,Downeaster,145,Downeaster,686,FAL,Falmouth,Maine,...,0,NaN,NaN,NaN,Falmouth,Muirfield Road at Railroad Crossing,NaN,04105,43.7696,-70.2595
51494,2022,3,State Supported,Downeaster,145,Downeaster,688,FAL,Falmouth,Maine,...,0,NaN,NaN,NaN,Falmouth,Muirfield Road at Railroad Crossing,NaN,04105,43.7696,-70.2595
51525,2022,3,State Supported,Downeaster,145,Downeaster,691,FAL,Falmouth,Maine,...,13,18.0,NaN,NaN,Falmouth,Muirfield Road at Railroad Crossing,NaN,04105,43.7696,-70.2595
51547,2022,3,State Supported,Downeaster,145,Downeaster,693,FAL,Falmouth,Maine,...,2,23.0,NaN,NaN,Falmouth,Muirfield Road at Railroad Crossing,NaN,04105,43.7696,-70.2595
51559,2022,3,State Supported,Downeaster,145,Downeaster,694,FAL,Falmouth,Maine,...,0,NaN,NaN,NaN,Falmouth,Muirfield Road at Railroad Crossing,NaN,04105,43.7696,-70.2595
51571,2022,3,State Supported,Downeaster,145,Downeaster,695,FAL,Falmouth,Maine,...,0,NaN,NaN,NaN,Falmouth,Muirfield Road at Railroad Crossing,NaN,04105,43.7696,-70.2595


#### 7.1.3 MCI

Formerly Amtrak's Michigan City, IN station, closed since April 2022. MCI row values can be
updated with the following information:

Amtrak Michigan City Station (closed)
100 Washington Street \
Michigan City, Indiana 46360 \
Latitude: `41.721111`, Longitude: `-86.905556`

In [28]:
values = ("Michigan City", "100 Washington Street", "46360", 41.721111, -86.905556)
mask = stations[COLS["station_code"]] == "MCI"
stations.loc[
    mask, [COLS["city"], COLS["address_01"], COLS["zip_code"], COLS["lat"], COLS["lon"]]
] = values
stations[mask]

,Fiscal Year,Fiscal Quarter,Service Line,Service,Route Miles,Sub Service,Train Number,Arrival Station Code,Arrival Station,State,...,Late Detraining Customers,Late Detraining Customers Avg Min Late,Code,Arrival Station Type,City,Address 01,Address 02,ZIP Code,Latitude,Longitude
52674,2022,3,State Supported,Michigan,299,Wolverine,350,MCI,Michigan City,Indiana,...,0,NaN,NaN,NaN,Michigan City,100 Washington Street,NaN,46360,41.721111,-86.905556
52717,2022,3,State Supported,Michigan,299,Wolverine,354,MCI,Michigan City,Indiana,...,0,NaN,NaN,NaN,Michigan City,100 Washington Street,NaN,46360,41.721111,-86.905556
52732,2022,3,State Supported,Michigan,299,Wolverine,355,MCI,Michigan City,Indiana,...,0,NaN,NaN,NaN,Michigan City,100 Washington Street,NaN,46360,41.721111,-86.905556
58106,2022,2,State Supported,Michigan,299,Wolverine,350,MCI,Michigan City,Indiana,...,11,39.0,NaN,NaN,Michigan City,100 Washington Street,NaN,46360,41.721111,-86.905556
58150,2022,2,State Supported,Michigan,299,Wolverine,354,MCI,Michigan City,Indiana,...,46,38.0,NaN,NaN,Michigan City,100 Washington Street,NaN,46360,41.721111,-86.905556
58165,2022,2,State Supported,Michigan,299,Wolverine,355,MCI,Michigan City,Indiana,...,71,57.0,NaN,NaN,Michigan City,100 Washington Street,NaN,46360,41.721111,-86.905556
63567,2022,1,State Supported,Michigan,299,Wolverine,350,MCI,Michigan City,Indiana,...,36,29.0,NaN,NaN,Michigan City,100 Washington Street,NaN,46360,41.721111,-86.905556
63610,2022,1,State Supported,Michigan,299,Wolverine,354,MCI,Michigan City,Indiana,...,31,41.0,NaN,NaN,Michigan City,100 Washington Street,NaN,46360,41.721111,-86.905556
63625,2022,1,State Supported,Michigan,299,Wolverine,355,MCI,Michigan City,Indiana,...,66,41.0,NaN,NaN,Michigan City,100 Washington Street,NaN,46360,41.721111,-86.905556


In [29]:
stations.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 68412 entries, 0 to 68411
Data columns (total 24 columns):
 #   Column                                  Non-Null Count  Dtype  
---  ------                                  --------------  -----  
 0   Fiscal Year                             68412 non-null  int64  
 1   Fiscal Quarter                          68412 non-null  int64  
 2   Service Line                            68412 non-null  object 
 3   Service                                 68412 non-null  object 
 4   Route Miles                             68412 non-null  int64  
 5   Sub Service                             68412 non-null  object 
 6   Train Number                            68412 non-null  int64  
 7   Arrival Station Code                    68412 non-null  object 
 8   Arrival Station                         68412 non-null  object 
 9   State                                   68412 non-null  object 
 10  Division                                68354 non-null  ob

## 8.0 Reorder columns

The `stations` columns are reordered as follows:

| Position | Column Name | Note |
| :----- | :------------- | :------------- |
| `0`-`1` | "Fiscal Year", "Fiscal Quarter" | &nbsp; |
| `2`-`5` | "Service Line", "Service", "Sub Service", "Train Number" | &nbsp; |
| `6`-`9` | "Arrival Station", "Arrival Station Type", "Code", "Arrival Station Code" | Drop "Code" after confirming column order. |
| `10`-`13` | "City", "Address 01", "Address 02", "ZIP Code" | &nbsp; |
| `14`-`17` | "State", "Division", "Region", "Country" | &nbsp; |
| `18`-`19` | "Latitude", "Longitude" | &nbsp; |
| `20`-`22` | "Total Detraining Customers", "Late Detraining Customers", "Late Detraining Customers Avg Min Late" | &nbsp; |

In [30]:
# Indices of interest
state_idx = stations.columns.get_loc(COLS["state"])
total_detrain_idx = stations.columns.get_loc(COLS["total_detrn"])
code_idx = stations.columns.get_loc("Code")

columns_start = stations.columns[:state_idx].tolist()
columns_start.extend([
    "Code",
    COLS["station_type"],
    COLS["city"],
    COLS["address_01"],
    COLS["address_02"],
    COLS["zip_code"],
])
print(f"columns_start = {columns_start}")

columns_middle = stations.columns[state_idx:total_detrain_idx].tolist()
columns_middle.extend([COLS["lat"], COLS["lon"]])
print(f"columns_middle = {columns_middle}")

columns_end = stations.columns[total_detrain_idx:code_idx].tolist()
print(f"columns_end = {columns_end}")

columns = columns_start + columns_middle + columns_end
print(f"columns = {columns}")

# Reorder DataFrame
stations = stations.loc[:, columns]
stations.shape

columns_start = ['Fiscal Year', 'Fiscal Quarter', 'Service Line', 'Service', 'Route Miles', 'Sub Service', 'Train Number', 'Arrival Station Code', 'Arrival Station', 'Code', 'Arrival Station Type', 'City', 'Address 01', 'Address 02', 'ZIP Code']
columns_middle = ['State', 'Division', 'Region', 'Country', 'Latitude', 'Longitude']
columns_end = ['Total Detraining Customers', 'Late Detraining Customers', 'Late Detraining Customers Avg Min Late']
columns = ['Fiscal Year', 'Fiscal Quarter', 'Service Line', 'Service', 'Route Miles', 'Sub Service', 'Train Number', 'Arrival Station Code', 'Arrival Station', 'Code', 'Arrival Station Type', 'City', 'Address 01', 'Address 02', 'ZIP Code', 'State', 'Division', 'Region', 'Country', 'Latitude', 'Longitude', 'Total Detraining Customers', 'Late Detraining Customers', 'Late Detraining Customers Avg Min Late']


(68412, 24)

In [31]:
stations.head(3)

,Fiscal Year,Fiscal Quarter,Service Line,Service,Route Miles,Sub Service,Train Number,Arrival Station Code,Arrival Station,Code,...,ZIP Code,State,Division,Region,Country,Latitude,Longitude,Total Detraining Customers,Late Detraining Customers,Late Detraining Customers Avg Min Late
0,2024,3,Long Distance,Auto Train,914,Auto Train,52,LOR,Lorton (Auto Train),LOR,...,22079,Virginia,South Atlantic,South,United States,38.708143,-77.220942,42445,23316,95.0
1,2024,3,Long Distance,Auto Train,914,Auto Train,53,SFA,Sanford (Auto Train),SFA,...,32771,Florida,South Atlantic,South,United States,28.808544,-81.291274,28034,18439,91.0
2,2024,3,Long Distance,California Zephyr,2408,California Zephyr,5,BRL,Burlington,BRL,...,52601,Iowa,West North Central,Midwest,United States,40.805788,-91.101951,557,223,54.0


## 9.0 Drop column [1 pt]

Drop the redundant "Code" column.

In [32]:
# YOUR CODE HERE
stations = stations.drop(columns=["Code"])

In [33]:
#hidden tests are within this cell

## 10.0 Late detraining passengers

Calculate the ratio of late detraining passengers to total detraining passengers _for each station_
and assign the results to a new column named "Late to Total Detraining Customers Ratio" (use the
associated `COLS` constant rather than hard-coding the string name ibnto the code). Round the 
values to the fitfh (`5th`) decimal place.

Note: Design your `lambda` function carefully to avoid a `ZeroDivisionError` error.

### 10.1 Calculate the percentage [1 pt]

In [38]:
# YOUR CODE HERE
target_column = "Late to Total Detraining Customers Ratio"
correct_key = [k for k, v in COLS.items() if v == target_column][0]

stations[COLS[correct_key]] = stations.apply(
    lambda row: round(row["Late Detraining Customers"] / row["Total Detraining Customers"], 5) 
    if row["Total Detraining Customers"] != 0 else 0.0, 
    axis=1
)

In [39]:
#hidden tests are within this cell

### 10.2 Sample the rows

Return a sample of rows to verify row values.

In [40]:
# Apply weights to sample (CBN stations are fewer)
weights = stations[COLS["svc_line"]].apply(lambda row: 3 if row == "Long Distance" else 1)
stations.sample(n=7, weights=weights, random_state=rdg)

,Fiscal Year,Fiscal Quarter,Service Line,Service,Route Miles,Sub Service,Train Number,Arrival Station Code,Arrival Station,Arrival Station Type,...,State,Division,Region,Country,Latitude,Longitude,Total Detraining Customers,Late Detraining Customers,Late Detraining Customers Avg Min Late,Late to Total Detraining Customers Ratio
5962,2024,3,State Supported,San Joaquins,372,San Joaquins,710,COC,Corcoran,Station Building (with waiting room),...,California,Pacific,West,United States,36.098516,-119.557050,355,179,43.0,0.50423
49695,2022,3,Northeast Corridor,Northeast Regional,519,On Spine Northeast Regional,168,BOS,Boston (South Station),Station Building (with waiting room),...,Massachusetts,New England,Northeast,United States,42.352311,-71.055304,1514,306,84.0,0.20211
54541,2022,2,Northeast Corridor,Acela Express,457,Acela Express,2166,NHV,New Haven (Union Station),Station Building (with waiting room),...,Connecticut,New England,Northeast,United States,41.297714,-72.926670,135,46,43.0,0.34074
48224,2022,3,Long Distance,Cardinal,1140,Cardinal,51,IND,Indianapolis,Station Building (with waiting room),...,Indiana,East North Central,Midwest,United States,39.761984,-86.160423,475,230,82.0,0.48421
27285,2023,3,Northeast Corridor,Northeast Regional,789,Richmond / Newport News / Norfolk,95,NWK,Newark (Penn Station),Station Building (with waiting room),...,New Jersey,Middle Atlantic,Northeast,United States,40.734706,-74.164750,556,64,33.0,0.11511
63639,2022,1,State Supported,Missouri,271,Missouri,313,HEM,Hermann,Station Building (with waiting room),...,Missouri,West North Central,Midwest,United States,38.707283,-91.432558,354,215,23.0,0.60734
1124,2024,3,Northeast Corridor,Acela,457,Acela,2251,BWI,BWI Thurgood Marshall Airport Station,Station Building (with waiting room),...,Maryland,South Atlantic,South,United States,39.192362,-76.694300,255,51,19.0,0.20000


### 10.3 Reorder columns [1 pt]

Move "Late to Total Detraining Customers Ratio" to the __second to last__ position in `stations`.

In [41]:
# YOUR CODE HERE
target_col = "Late to Total Detraining Customers Ratio"
cols = stations.columns.tolist()
cols.remove(target_col)
cols.insert(-1, target_col)
stations = stations[cols]

In [42]:
#hidden tests are within this cell

## 11.0 Persist data

### 11.1 Recheck data.

In [43]:
stations.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 68412 entries, 0 to 68411
Data columns (total 24 columns):
 #   Column                                    Non-Null Count  Dtype  
---  ------                                    --------------  -----  
 0   Fiscal Year                               68412 non-null  int64  
 1   Fiscal Quarter                            68412 non-null  int64  
 2   Service Line                              68412 non-null  object 
 3   Service                                   68412 non-null  object 
 4   Route Miles                               68412 non-null  int64  
 5   Sub Service                               68412 non-null  object 
 6   Train Number                              68412 non-null  int64  
 7   Arrival Station Code                      68412 non-null  object 
 8   Arrival Station                           68412 non-null  object 
 9   Arrival Station Type                      68386 non-null  object 
 10  City                              

### 11.2 Write to file. [1 pt]

Write data to a CSV file.

In [46]:
filepath = "station_performance_metrics-v1p2.csv"

stations.to_csv(filepath, index=False)

In [47]:
#hidden tests are within this cell

## 12.0 Watermark

In [48]:
%load_ext watermark
%watermark -h -i -iv -m -v

The watermark extension is already loaded. To reload it, use:
  %reload_ext watermark
Python implementation: CPython
Python version       : 3.11.9
IPython version      : 8.26.0

Compiler    : GCC 12.3.0
OS          : Linux
Release     : 6.5.0-1024-aws
Machine     : x86_64
Processor   : x86_64
CPU cores   : 32
Architecture: 64bit

Hostname: 97740255e5fb

pandas: 2.2.3
numpy : 2.1.3
json  : 2.0.9
re    : 2.2.1

